In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

training_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
])


training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=training_transform
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=transforms.ToTensor()
)

batch_size = 64
training_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)


100%|██████████| 26.4M/26.4M [00:01<00:00, 18.7MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 305kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.54MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 12.4MB/s]


In [ ]:

device = torch.device("cpu")
print(f"Using {device} device")


Using cpu device


In [ ]:
class Net(nn.Module):
  def __init__(self):
    super(Net, self).__init__()
    self.flatten = nn.Flatten()
    self.linear_relu_stack = nn.Sequential(
        nn.Linear(28*28, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, 10),
    )

  def forward(self, x):
    x = self.flatten(x)
    logits = self.linear_relu_stack(x)
    return logits

model = Net().to(device)
print(model)

Net(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=512, out_features=512, bias=True)
    (5): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.5, inplace=False)
    (8): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2)

def train(dataloader, model, loss_fn, optimizer, scheduler):
    size = len(dataloader.dataset)
    model.train()
    running_loss = 0.0

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        running_loss += loss.item()

        if batch % 100 == 0:
            current = batch * len(X)
            print(f"loss: {loss.item():>7f}  [{current:>5d}/{size:>5d}]")

    avg_loss = running_loss / len(dataloader)
    scheduler.step(avg_loss)  # LR scheduler usually uses *average* loss

def test(dataloader, model, loss_fn, scheduler=None):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0

    with torch.no_grad():  # Better style than inference_mode() for compatibility
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    accuracy = correct / size

    if scheduler:  # Scheduler step only if passed in
        scheduler.step(test_loss)

    print(f"Test Error: \n Accuracy: {(100*accuracy):>0.1f}%, Avg loss: {test_loss:>8f} \n")


In [ ]:
if __name__ == "__main__":

    epochs = 8

    for t in range(epochs):
        print(f"Epoch {t+1}\n-------------------------------")
        train(training_dataloader, model, loss_fn, optimizer, scheduler)
        test(test_dataloader, model, loss_fn, scheduler)

    print("Done!")

    torch.save(model.state_dict(), "fashion_mnist.pth")
    print("Saved PyTorch Model State")

Epoch 1
-------------------------------
loss: 2.379772  [    0/60000]
loss: 0.910086  [ 6400/60000]
loss: 0.638869  [12800/60000]
loss: 0.859963  [19200/60000]
loss: 0.716841  [25600/60000]
loss: 0.888801  [32000/60000]
loss: 0.748933  [38400/60000]
loss: 0.704304  [44800/60000]
loss: 0.789579  [51200/60000]
loss: 0.887560  [57600/60000]
Test Error: 
 Accuracy: 77.6%, Avg loss: 0.584451 

Epoch 2
-------------------------------
loss: 0.700050  [    0/60000]
loss: 0.694504  [ 6400/60000]
loss: 0.527274  [12800/60000]
loss: 0.711794  [19200/60000]
loss: 0.740331  [25600/60000]
loss: 0.732972  [32000/60000]
loss: 0.548222  [38400/60000]
loss: 0.603344  [44800/60000]
loss: 0.824308  [51200/60000]
loss: 0.684800  [57600/60000]
Test Error: 
 Accuracy: 78.9%, Avg loss: 0.548652 

Epoch 3
-------------------------------
loss: 0.607969  [    0/60000]
loss: 0.652993  [ 6400/60000]
loss: 0.459733  [12800/60000]
loss: 0.776461  [19200/60000]
loss: 0.787824  [25600/60000]
loss: 0.574903  [32000/600

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from torchvision import datasets, transforms
# from torchvision.transforms import ToTensor, Lambda, Compose
import matplotlib.pyplot as plt
from PIL import Image


device = torch.device("cpu")

model = Net().to(device)
print(model)


model = Net().to(device)
model.load_state_dict(torch.load("fashion_mnist.pth"))
model.eval()

image_path = f"test.png"
image = Image.open(image_path).convert("L")

transform  = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((28, 28)),

])

image = transform(image).unsqueeze(0).to(device)

with torch.no_grad():
  output = model(image)
  prediction = output.argmax(1).item()

print(f"Prediction: {prediction}")



Net(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=512, out_features=512, bias=True)
    (5): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.5, inplace=False)
    (8): Linear(in_features=512, out_features=10, bias=True)
  )
)
Prediction: 9
